# Feature Engineering
Une `transactions`, `members_v3` y `user_logs` en una sola matriz de features por usuario.  
Salida: `data/processed/features_train.parquet`

| Fuente | Features | Cobertura |
|---|---|---|
| transactions | 14 | 100% |
| members_v3 | 6 | ~88% |
| user_logs | 7 | ~88% |

> La primera ejecución agrega user_logs (~10 min). Las siguientes usan el caché en `data/processed/user_logs_agg.parquet`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.features.build_features_05 import (
    build_and_save,
    get_transaction_features,
    build_member_features,
    get_log_features,
    join_all,
    impute_missing,
    DATA_RAW,
    FEATURES_OUT,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Construir y guardar feature matrix

In [ ]:
df = build_and_save()

## 2. Inspección del dataset final

In [ ]:
print(f'Shape: {df.shape}')
print(f'Churn rate: {df["is_churn"].mean():.2%}')
df.head()

In [ ]:
df.describe().T

In [ ]:
# Nulos restantes
nulls = df.isnull().sum()
print('Nulos por columna:')
print(nulls[nulls > 0] if nulls.any() else 'Ninguno ✓')

## 3. Cobertura de las distintas fuentes

In [ ]:
labels = pd.read_csv(DATA_RAW / 'train.csv')[['msno', 'is_churn']]
total = len(labels)

coverage = {
    'Transactions': df['n_transactions'].notna().sum(),
    'Members': df['has_member_record'].sum(),
    'User logs': df['has_log_record'].sum(),
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(coverage.keys(), [v / total * 100 for v in coverage.values()],
              color=['#2196F3', '#4CAF50', '#9C27B0'], edgecolor='white')
for bar, (k, v) in zip(bars, coverage.items()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{v:,}\n({v/total:.1%})', ha='center', fontsize=10)
ax.set_ylim(0, 115)
ax.set_ylabel('% de usuarios de train')
ax.set_title('Cobertura por fuente de datos')
plt.tight_layout()
plt.show()

## 4. Distribuciones de features clave

In [ ]:
KEY_FEATURES = [
    'last_is_cancel', 'last_is_auto_renew', 'n_transactions',
    'days_since_last', 'listening_trend', 'tenure_days',
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, KEY_FEATURES):
    lo, hi = df[col].quantile([0.01, 0.99])
    data = df[df[col].between(lo, hi)]
    renewal = data[data['is_churn'] == 0][col]
    churn = data[data['is_churn'] == 1][col]
    ax.hist(renewal, bins=50, alpha=0.6, color='#4CAF50', label='Renewal', density=True)
    ax.hist(churn, bins=50, alpha=0.6, color='#F44336', label='Churn', density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.suptitle('Top features: Churn vs Renewal', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlación con el target

In [ ]:
feature_cols = [c for c in df.columns if c not in ['msno', 'is_churn']]
corr = df[feature_cols + ['is_churn']].corr()['is_churn'].drop('is_churn')
corr_sorted = corr.abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#F44336' if corr[f] > 0 else '#4CAF50' for f in corr_sorted.index]
ax.barh(corr_sorted.index[::-1], corr_sorted.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_title('Correlación absoluta con is_churn')
ax.set_xlabel('|Pearson r|')
plt.tight_layout()
plt.show()

print('\nTop 10 features por correlación con is_churn:')
print(corr.reindex(corr_sorted.index).head(10).to_string())

## 6. Matriz de correlación entre features

In [ ]:
top_features = corr_sorted.head(15).index.tolist()
corr_matrix = df[top_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, ax=ax, annot_kws={'size': 8}
)
ax.set_title('Correlación entre top 15 features')
plt.tight_layout()
plt.show()

## 7. Resumen del dataset

El parquet está en `data/processed/features_train.parquet` y es el input directo para el modelado.

In [ ]:
print('═' * 60)
print('FEATURE MATRIX — RESUMEN')
print('═' * 60)
print(f'  Usuarios         : {len(df):,}')
print(f'  Features         : {len(feature_cols)}')
print(f'  Churn rate       : {df["is_churn"].mean():.2%}')
print(f'  Con registro members: {df["has_member_record"].mean():.1%}')
print(f'  Con registro logs   : {df["has_log_record"].mean():.1%}')
print(f'  Nulos restantes  : {df.isnull().sum().sum()}')
print(f'  Guardado en      : data/processed/features_train.parquet')